In [83]:
# !pip -q install chromadb

In [79]:
import chromadb
import pandas as pd
import tqdm as tq
from utils import get_examples_from_df
import json
import re
from pathlib import Path
from datetime import datetime as dt

In [52]:
base_path = Path().cwd()

In [41]:
chroma_client = chromadb.Client()

# switch `create_collection` to `get_or_create_collection` to avoid creating a new collection every time
collection = chroma_client.get_or_create_collection(
    name="requirements",
    metadata={"hnsw:space": "cosine"} # cosine, l2
)

In [12]:
df = pd.read_excel("data/requirements.xlsx")

In [7]:
indexes_to_drop, examples = get_examples_from_df(df, 5)

In [42]:
documents = []
metadatas = []

for e1 in examples.values():
    for e2 in e1:
        documents.append(e2[0])
        metadatas.append({"vector": e2[1]})

In [43]:
# switch `add` to `upsert` to avoid adding the same documents every time
collection.upsert(
    ids=[f"id_{i}"for i in range(len(documents))],
    documents=documents,
    metadatas=metadatas
)

In [15]:
df_reduced = df.drop(index=indexes_to_drop)

In [73]:
def construct_examples_text_from_chroma_db(collection, query, n_examples=5):
    results = collection.query(
		query_texts=[query],
		n_results=n_examples
	)
    # join all examples in a text format to add to prompt
    examples_txt = ""
    
    for e in zip(results["documents"][0],results["metadatas"][0]):
        examples_txt += f"Requirement: {e[0]}\n"
        examples_txt += f"Vector: {e[1]['vector']}\n"
        examples_txt += "\n"
		
    return results, examples_txt

# LLM

In [49]:
from langchain_openai import AzureChatOpenAI

from prompts.SystemPrompts import SystemPrompt
from prompts.Sensors import Sensors
from prompts.UserPrompt import UserPrompt

In [74]:
# llm
llm = AzureChatOpenAI(
    deployment_name="gpt-35",
    temperature=0.0
)

# Run for all Requirements

In [75]:
N_EXAMPLES = 5

def parse_result(res):
    '''Parse the LLM result'''
    pattern = r"\[(.*?)\]"
    vec = re.findall(pattern, res)[0]
    return f"[{vec}]".replace(" ", "")


def run_all_reqs(instance):

    result = {}

    result["idx"] = instance[0]
    result["requirement"] = instance[1].iloc[0]
    result["true_vector"] = "[" + ",".join(map(str, instance[1].iloc[1:])) + "]"
    
    results_rag, examples_txt = construct_examples_text_from_chroma_db(collection, result["requirement"], N_EXAMPLES)
    
    # system prompt
    messages = [
        {'role': 'system',
        'content': SystemPrompt.format(sensors=Sensors,examples=examples_txt)}
    ]

    # add user prompt
    messages.append({"role":"user", "content":UserPrompt.format(req=result["requirement"])})

    # run LLM
    response = llm.invoke(messages)
    result["ai_response"] = response.content
    result["pred_vector"] = parse_result(result["ai_response"])

    result["accuracy"] = result["pred_vector"] == result["true_vector"]

    result["ai_token_usage"] = response.response_metadata["token_usage"]

    return result, results_rag

In [91]:
result = [run_all_reqs(i) for i in tq.tqdm(df.iterrows())]

235it [02:24,  1.62it/s]


In [92]:
results_rag = [r[1] for r in result]
results = [r[0] for r in result]

In [94]:
accuracy = 0
total_tokens = 0
total_completion_tokens = 0

for r in results:
    accuracy += r["accuracy"]
    total_tokens += r["ai_token_usage"]["total_tokens"]
    total_completion_tokens += r["ai_token_usage"]["completion_tokens"]

number_of_reqs = len(results)
accuracy /= len(results)
avg_token_per_req = total_tokens / len(results)
avg_completion_token_per_req = total_completion_tokens / len(results)

In [95]:
accuracy

0.8212765957446808

In [98]:
# save results
time = dt.now()

results_path = "results/conv_{model}_rag_n-{examples}_acc-{accuracy}_{time}.json"
results_path = results_path.format(
    model=llm.deployment_name,
    examples=N_EXAMPLES,
    time=time.strftime('%m.%d.%Y-%H:%M:%S'),
    accuracy=round(accuracy, 3)
)

results_file = base_path / results_path
results_file.parent.mkdir(exist_ok=True)
results_file.touch()

with results_file.open("w") as f:
    json.dump({"accuracy": accuracy,
        "number_of_reqs": number_of_reqs,
        "total_tokens": total_tokens,
        "total_completion_tokens": total_completion_tokens,
        "avg_token_per_req": avg_token_per_req,
        "avg_completion_token_per_req": avg_completion_token_per_req,
        "responses": results}, f, indent=4)